# Function Calling with IBM Granite 4

- <https://www.ibm.com/new/announcements/ibm-granite-4-0-hyper-efficient-high-performance-hybrid-models>
- <https://huggingface.co/ibm-granite/granite-4.0-h-micro>

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
device = "mps"
model_path = "ibm-granite/granite-4.0-h-micro"
# "ibm-granite/granite-4.0-h-small"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path, device_map=device)

The fast path is not available because one of `(selective_state_update, causal_conv1d_fn, causal_conv1d_update)` is None. Falling back to the naive implementation. To install follow https://github.com/state-spaces/mamba/#installation and https://github.com/Dao-AILab/causal-conv1d


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [2]:
tokenizer

GPT2TokenizerFast(name_or_path='ibm-granite/granite-4.0-h-micro', vocab_size=100352, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<|end_of_text|>', 'eos_token': '<|end_of_text|>', 'unk_token': '<|unk|>', 'pad_token': '<|pad|>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	100256: AddedToken("<|pad|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100257: AddedToken("<|end_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100258: AddedToken("<|fim_prefix|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=False),
	100259: AddedToken("<|fim_middle|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=False),
	100260: AddedToken("<|fim_suffix|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=False),
	100261: AddedToken("<|fim_pad

## Chat Template

In [3]:
print(tokenizer.get_chat_template())

{%- set tools_system_message_prefix = 'You are a helpful assistant with access to the following tools. You may call one or more tools to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>'  %}
{%- set tools_system_message_suffix = '\n</tools>\n\nFor each tool call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call>. If a tool does not exist in the provided list of tools, notify the user that you do not have the ability to fulfill the request.' %}
{%- set documents_system_message_prefix = 'You are a helpful assistant with access to the following documents. You may use one or more documents to assist with the user query.\n\nYou are given a list of documents within <documents></documents> XML tags:\n<documents>' %}
{%- set documents_system_message_suffix = '\n</documents>\n\nWrite the response

In [4]:
model.eval()
# change input text as desired
chat = [
    { "role": "user", "content": "How is the weather in London tomorrow?" },
]
chat = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
chat

'<|start_of_role|>system<|end_of_role|>You are a helpful assistant. Please ensure responses are professional, accurate, and safe.<|end_of_text|>\n<|start_of_role|>user<|end_of_role|>How is the weather in London tomorrow?<|end_of_text|>\n<|start_of_role|>assistant<|end_of_role|>'

In [5]:
# tokenize the text
input_tokens = tokenizer(chat, return_tensors="pt").to(device)

In [6]:
# generate output tokens
output = model.generate(**input_tokens, 
                        max_new_tokens=100)
# decode output tokens into text
output = tokenizer.batch_decode(output)
# print output
print(output[0])

/opt/homebrew/anaconda3/envs/ws25_5/lib/python3.12/site-packages/torch/nn/functional.py:5294: UserWarning: MPS: The constant padding of more than 3 dimensions is not currently supported natively. It uses View Ops default implementation to run. This may have performance implications. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/mps/operations/Pad.mm:468.)
  return torch._C._nn.pad(input, pad, mode, value)


<|start_of_role|>system<|end_of_role|>You are a helpful assistant. Please ensure responses are professional, accurate, and safe.<|end_of_text|>
<|start_of_role|>user<|end_of_role|>How is the weather in London tomorrow?<|end_of_text|>
<|start_of_role|>assistant<|end_of_role|>As an AI, I don't have real-time capabilities to provide current or future weather updates. I recommend checking a reliable weather forecasting website or app for the most accurate information.<|end_of_text|>


## Function Calling

In [7]:
tools = [
    {
        "name": "get_weather",
        "description": "Returns the weather for date and location",
        "parameters": {
            "type": "object",
            "properties": {
                "date": {"type": "string"},
                "location": {"type": "string"}
            },
            "required": ["date", "location"]
        }
    }
]

In [8]:
# add tools to chat template
chat = [
    { "role": "user", "content": "How is the weather in London tomorrow?" },
]
chat = tokenizer.apply_chat_template(chat, tools=tools, tokenize=False, add_generation_prompt=True)

In [10]:
print(chat)

<|start_of_role|>system<|end_of_role|>You are a helpful assistant with access to the following tools. You may call one or more tools to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"name": "get_weather", "description": "Returns the weather for date and location", "parameters": {"type": "object", "properties": {"date": {"type": "string"}, "location": {"type": "string"}}, "required": ["date", "location"]}}
</tools>

For each tool call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call>. If a tool does not exist in the provided list of tools, notify the user that you do not have the ability to fulfill the request.<|end_of_text|>
<|start_of_role|>user<|end_of_role|>How is the weather in London tomorrow?<|end_of_text|>
<|start_of_role|>assistant<|end_of_role|>


In [11]:
# tokenize the text
input_tokens = tokenizer(chat, return_tensors="pt").to(device)
# generate output tokens
output = model.generate(**input_tokens, 
                        max_new_tokens=100)
# decode output tokens into text
output = tokenizer.batch_decode(output)
# print output
print(output[0])

<|start_of_role|>system<|end_of_role|>You are a helpful assistant with access to the following tools. You may call one or more tools to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"name": "get_weather", "description": "Returns the weather for date and location", "parameters": {"type": "object", "properties": {"date": {"type": "string"}, "location": {"type": "string"}}, "required": ["date", "location"]}}
</tools>

For each tool call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call>. If a tool does not exist in the provided list of tools, notify the user that you do not have the ability to fulfill the request.<|end_of_text|>
<|start_of_role|>user<|end_of_role|>How is the weather in London tomorrow?<|end_of_text|>
<|start_of_role|>assistant<|end_of_role|><tool_call>
{"name": "get_weather", "ar

In [12]:
tools = [{
  "name": "get_weather",
  "description": "Get weather forecast for a specific date and location.",
  "parameters": {
    "type": "object",
    "properties": {
      "location": {"type": "string"},
      "date": {
        "type": "string",
        "description": "An ISO date (YYYY-MM-DD). Convert relative dates like 'tomorrow' before filling this field."
      }
    },
    "required": ["location", "date"]
  }
}
]

In [13]:
# change input text as desired
from datetime import datetime
current_date = datetime.now().strftime("%Y-%m-%d")
chat = [
    { "role": "system", "content": f'Assume todays date is {current_date}' },
    { "role": "user", "content": "How is the weather in London tomorrow?" },
]
chat = tokenizer.apply_chat_template(chat, tools=tools, tokenize=False, add_generation_prompt=True)

In [14]:
print(chat)

<|start_of_role|>system<|end_of_role|>Assume todays date is 2025-11-28

You are a helpful assistant with access to the following tools. You may call one or more tools to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"name": "get_weather", "description": "Get weather forecast for a specific date and location.", "parameters": {"type": "object", "properties": {"location": {"type": "string"}, "date": {"type": "string", "description": "An ISO date (YYYY-MM-DD). Convert relative dates like 'tomorrow' before filling this field."}}, "required": ["location", "date"]}}
</tools>

For each tool call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call>. If a tool does not exist in the provided list of tools, notify the user that you do not have the ability to fulfill the request.<|end_of_text|>
<|start_of_ro

In [15]:
# tokenize the text
input_tokens = tokenizer(chat, return_tensors="pt").to(device)
# generate output tokens
output = model.generate(**input_tokens, 
                        max_new_tokens=100)
# decode output tokens into text
output = tokenizer.batch_decode(output)
# print output
print(output[0])

<|start_of_role|>system<|end_of_role|>Assume todays date is 2025-11-28

You are a helpful assistant with access to the following tools. You may call one or more tools to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"name": "get_weather", "description": "Get weather forecast for a specific date and location.", "parameters": {"type": "object", "properties": {"location": {"type": "string"}, "date": {"type": "string", "description": "An ISO date (YYYY-MM-DD). Convert relative dates like 'tomorrow' before filling this field."}}, "required": ["location", "date"]}}
</tools>

For each tool call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call>. If a tool does not exist in the provided list of tools, notify the user that you do not have the ability to fulfill the request.<|end_of_text|>
<|start_of_ro